<a href="https://colab.research.google.com/github/abhilash-20/demo_tts/blob/Main/distilBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
!pip install transformers datasets evaluate accelerate


In [ ]:
!pip install scikit-learn pandas numpy


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


GPU available: True
GPU name: Tesla T4


In [ ]:
!pip install seqeval


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=18b71c055b017186a0a9b8287e48adc196a6376921e318b0a3e15d7047e7f050
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
# --- IMPORTS ---
import pandas as pd
from datasets import Dataset
import evaluate  # ✅ replaces deprecated load_metric
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
import numpy as np
from sklearn.model_selection import train_test_split as sk_train_test_split

# --- CONFIGURATION ---
NER_CSV_PATH = "/content/ner_training_data_augmented.csv"  # ✅ your dataset file
model_name = "distilbert-base-uncased"
metric = evaluate.load("seqeval")  # ✅ new evaluate library
output_dir = "./distilbert_ner_results"
# ---------------------

# --- 1. LOAD DATA ---
ner_df = pd.read_csv("/content/ner_training_data_augmented.csv")

# The file should have columns: sentence_id | words | tags
print(ner_df.head())

# --- 2. GROUP BY SENTENCE ---
agg_func = lambda s: [(w, t) for w, t in zip(s["words"].tolist(), s["tags"].tolist())]
grouped_sentences = ner_df.groupby("sentence_id").apply(agg_func).tolist()

# --- 3. TAG MAPPINGS ---
unique_tags = sorted(list(set(ner_df["tags"].values)))
tag_to_id = {t: i for i, t in enumerate(unique_tags)}
id_to_tag = {i: t for t, i in tag_to_id.items()}

# --- 4. CREATE TOKEN/TAG LISTS ---
all_tokens = [[item[0] for item in sentence] for sentence in grouped_sentences]
all_ner_tags = [[tag_to_id[item[1]] for item in sentence] for sentence in grouped_sentences]

# --- 5. CONVERT TO DATASET ---
dataset = Dataset.from_dict({
    "id": list(range(len(all_tokens))),
    "tokens": all_tokens,
    "ner_tags": all_ner_tags
})

# --- 6. TRAIN/TEST SPLIT ---
train_test_split_dict = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split_dict["train"]
eval_dataset = train_test_split_dict["test"]

print(f"✅ Total Sentences: {len(dataset)}")
print(f"✅ Unique Tags: {unique_tags}")


   sentence_id      words tags
0            1        The    O
1            1        new    O
2            1    project    O
3            1        was    O
4            1  initiated    O
✅ Total Sentences: 1600
✅ Unique Tags: ['B-PER', 'I-PER', 'O']


/tmp/ipython-input-2171160687.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_sentences = ner_df.groupby("sentence_id").apply(agg_func).tolist()


In [ ]:
# --- TOKENIZATION ---
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_eval = eval_dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/1280 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

In [ ]:
# --- MODEL ---
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(unique_tags),
    id2label=id_to_tag,
    label2id=tag_to_id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# --- METRICS FUNCTION ---
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[id_to_tag[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id_to_tag[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# --- TRAINING ARGUMENTS ---
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=25,
    warmup_ratio=0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"  # disable wandb
)

# --- TRAINER ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# --- TRAIN ---
trainer.train()


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-922305400.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.210800,0.056909,0.942735,0.943541,0.943138,0.982459
2,0.004700,0.000817,1.000000,1.000000,1.000000,1.000000
3,0.000700,0.000279,1.000000,1.000000,1.000000,1.000000
4,0.000400,0.000171,1.000000,1.000000,1.000000,1.000000
5,0.002300,0.000260,1.000000,1.000000,1.000000,1.000000
6,0.000700,0.000121,1.000000,1.000000,1.000000,1.000000
7,0.000200,0.000087,1.000000,1.000000,1.000000,1.000000
8,0.000100,0.000075,1.000000,1.000000,1.000000,1.000000
9,0.000100,0.000064,1.000000,1.000000,1.000000,1.000000
10,0.000100,0.000057,1.000000,1.000000,1.000000,1.000000


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("./distilbert_ner_model")
tokenizer.save_pretrained("./distilbert_ner_model")


('./distilbert_ner_model/tokenizer_config.json',
 './distilbert_ner_model/special_tokens_map.json',
 './distilbert_ner_model/vocab.txt',
 './distilbert_ner_model/added_tokens.json',
 './distilbert_ner_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_path = "/content/distilbert_ner_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

ner_pipe = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")


Device set to use cuda:0


In [ ]:
import pandas as pd

# Load your existing dataset
df = pd.read_csv("/content/ner_training_data.csv")

# Expanded list of example name pairs (50 total)
new_sentences = [
    ("Barack", "Obama"),
    ("Elon", "Musk"),
    ("Taylor", "Swift"),
    ("Lionel", "Messi"),
    ("Cristiano", "Ronaldo"),
    ("Virat", "Kohli"),
    ("Ariana", "Grande"),
    ("Bill", "Gates"),
    ("Mark", "Zuckerberg"),
    ("Narendra", "Modi"),
    ("Amitabh", "Bachchan"),
    ("Priyanka", "Chopra"),
    ("Shah", "Rukh"),
    ("Salman", "Khan"),
    ("Alia", "Bhatt"),
    ("Deepika", "Padukone"),
    ("Ranbir", "Kapoor"),
    ("Katrina", "Kaif"),
    ("Sundar", "Pichai"),
    ("Satya", "Nadella"),
    ("Jeff", "Bezos"),
    ("Tim", "Cook"),
    ("Steve", "Jobs"),
    ("Ratan", "Tata"),
    ("Mukesh", "Ambani"),
    ("Gautam", "Adani"),
    ("Roger", "Federer"),
    ("Serena", "Williams"),
    ("Novak", "Djokovic"),
    ("Emma", "Watson"),
    ("Daniel", "Radcliffe"),
    ("Jennifer", "Lawrence"),
    ("Chris", "Evans"),
    ("Robert", "Downey"),
    ("Scarlett", "Johansson"),
    ("Zendaya", "Coleman"),
    ("Tom", "Holland"),
    ("Greta", "Thunberg"),
    ("Malala", "Yousafzai"),
    ("A. P. J.", "Abdul Kalam"),
    ("Kiran", "Bedi"),
    ("Saina", "Nehwal"),
    ("Mary", "Kom"),
    ("Neeraj", "Chopra"),
    ("PV", "Sindhu"),
    ("Rahul", "Dravid"),
    ("Sourav", "Ganguly"),
    ("MS", "Dhoni"),
    ("Rohit", "Sharma"),
    ("Sachin", "Tendulkar"),
    ("Hardik", "Pandya"),
]

# Build artificial NER entries with slight sentence variations
import random

templates = [
    "{} {} visited India today.",
    "{} {} gave a speech in Delhi.",
    "{} {} met with the Prime Minister.",
    "{} {} launched a new project.",
    "{} {} attended a conference in Mumbai.",
    "{} {} received an award.",
    "{} {} is a famous personality.",
    "{} {} announced their new venture.",
    "{} {} played exceptionally well today.",
    "{} {} was seen in New York.",
]

new_rows = []
sentence_id = df["sentence_id"].max() + 1

for first, last in new_sentences:
    template = random.choice(templates)
    sentence = template.format(first, last).split()
    tags = []
    for i, word in enumerate(sentence):
        if word == first:
            tags.append("B-PER")
        elif word == last:
            tags.append("I-PER")
        else:
            tags.append("O")
    for w, t in zip(sentence, tags):
        new_rows.append({"sentence_id": sentence_id, "words": w, "tags": t})
    sentence_id += 1

# Combine and save
df_aug = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
df_aug.to_csv("/content/ner_training_data_augmented.csv", index=False)

print("✅ New dataset saved as ner_training_data_augmented.csv")
print(df_aug["tags"].value_counts())


✅ New dataset saved as ner_training_data_augmented.csv
tags
O        1183
B-PER      80
I-PER      52
Name: count, dtype: int64


In [ ]:
import random
import pandas as pd

# ----------------------------
# STEP 1: Define name lists
# ----------------------------
first_names = ["Elon", "Virat", "Barack", "Marie", "Steve", "Amit", "Sundar", "Greta",
               "Emma", "Ravi", "Priya", "John", "Sophia", "David", "Neha", "Leonardo",
               "Natalie", "Daniel", "Selena", "Hrithik", "Ranbir", "Alia", "Deepika", "Taylor"]
last_names = ["Musk", "Kohli", "Obama", "Curie", "Jobs", "Sharma", "Pichai", "Thunberg",
              "Watson", "Patel", "Gupta", "Singh", "Roy", "Das", "Mehta", "DiCaprio",
              "Portman", "Craig", "Gomez", "Roshan", "Kapoor", "Bhatt", "Padukone", "Swift"]

def random_name():
    return f"{random.choice(first_names)} {random.choice(last_names)}"

# ----------------------------
# STEP 2: Sentence templates
# ----------------------------
templates = [
    "{PER1} met {PER2} during the conference in Paris.",
    "When {PER1} arrived, {PER2} and {PER3} were already discussing the plan.",
    "Dr. {PER1} and Mr. {PER2} presented their findings together.",
    "{PER1}'s collaboration with {PER2} has been highly successful.",
    "{PER1}, along with {PER2} and {PER3}, attended the award ceremony.",
    "Rumor has it that {PER1} will replace {PER2} as CEO while {PER3} manages operations.",
    "Before {PER1} could respond, {PER2} interrupted with a counter-argument.",
    "The new project was initiated by {PER1} under the supervision of {PER2}.",
    "Both {PER1} and {PER2} contributed to the discovery published by {PER3}.",
    "{PER1} thanked {PER2} for the support given during the expedition.",
    "According to {PER1}, {PER2} has been mentoring {PER3} for years.",
    "The film directed by {PER1} features {PER2} and {PER3} in lead roles.",
    "{PER1} and {PER2}'s joint venture received praise from {PER3}.",
    "{PER1}'s assistant mentioned that {PER2} would join {PER3} soon.",
    "After the meeting, {PER1} called {PER2} to discuss {PER3}'s proposal.",
    "While {PER1} worked on the design, {PER2} handled the finances and {PER3} reviewed progress.",
    "{PER1} sent a letter to {PER2}, which was later shared by {PER3}.",
    "The friendship between {PER1} and {PER2} impressed {PER3}.",
    "{PER1} reported that {PER2} was last seen with {PER3}.",
    "Even though {PER1} disagreed, {PER2} convinced {PER3} to proceed."
]

# ----------------------------
# STEP 3: Non-person templates (all O)
# ----------------------------
neutral_templates = [
    "The weather in London was pleasant yesterday.",
    "Many students attended the online workshop on AI ethics.",
    "The project deadline was extended by two weeks.",
    "A group of researchers discovered a new species of butterfly.",
    "Technology continues to evolve at a rapid pace.",
    "The car broke down near the highway exit.",
    "Everyone enjoyed the movie screening last night.",
    "Coffee is the most popular beverage in the office.",
    "A massive storm caused flooding in several areas.",
    "The dog chased the cat around the garden.",
    "People often underestimate the power of kindness.",
    "The meeting was postponed to next Monday.",
    "The computer lab will remain closed for maintenance.",
    "Global warming remains one of the greatest challenges today.",
    "She forgot her umbrella at the restaurant.",
    "The train arrived ten minutes late.",
    "Books provide a window to different worlds.",
    "The museum reopened after major renovations.",
    "Music brings people together across cultures.",
    "Everyone clapped as the fireworks lit up the sky."
]

# ----------------------------
# STEP 4: Label generator
# ----------------------------
def label_sentence(sentence, people=None):
    tokens = sentence.replace('.', ' .').replace(',', ' ,').split()
    labels = []
    for token in tokens:
        label = "O"
        if people:
            for name in people.values():
                parts = name.split()
                if token == parts[0]:
                    label = "B-PER"
                elif len(parts) > 1 and token == parts[1]:
                    label = "I-PER"
        labels.append(label)
    return list(zip(tokens, labels))

# ----------------------------
# STEP 5: Generate dataset
# ----------------------------
dataset = []
sentence_id = 1

# 1500 sentences with names
for _ in range(1500):
    per1, per2, per3 = random_name(), random_name(), random_name()
    sent = random.choice(templates).format(PER1=per1, PER2=per2, PER3=per3)
    labeled = label_sentence(sent, {"PER1": per1, "PER2": per2, "PER3": per3})
    for word, tag in labeled:
        dataset.append({"sentence_id": sentence_id, "word": word, "tag": tag})
    sentence_id += 1

# 100 neutral sentences with no PER tags
for _ in range(100):
    sent = random.choice(neutral_templates)
    labeled = label_sentence(sent, None)
    for word, tag in labeled:
        dataset.append({"sentence_id": sentence_id, "word": word, "tag": tag})
    sentence_id += 1

# ----------------------------
# STEP 6: Save as CSV
# ----------------------------
df = pd.DataFrame(dataset)
df.to_csv("ner_training_data_augmented.csv", index=False, encoding="utf-8")

print("✅ Saved as ner_training_data_augmented.csv")
print(df["tag"].value_counts())
print(f"Total sentences: {sentence_id - 1}, Total tokens: {len(df)}")


✅ Saved as ner_training_data_augmented.csv
tag
O        14666
B-PER     4046
I-PER     3694
Name: count, dtype: int64
Total sentences: 1600, Total tokens: 22406


In [ ]:
def merge_entities_split(preds):
    merged = []
    current = {"entity_group": None, "word": "", "score": []}
    last_end = -1

    for p in preds:
        # Handle both possible keys: "entity" or "entity_group"
        entity_tag = p.get("entity", p.get("entity_group"))
        word_piece = p["word"].replace("##", "")
        score = float(p["score"])

        if entity_tag == "O" or entity_tag is None:
            if current["entity_group"]:
                current["score"] = sum(current["score"]) / len(current["score"])
                merged.append(current)
                current = {"entity_group": None, "word": "", "score": []}
            continue

        entity = entity_tag.split("-")[-1]
        if current["entity_group"] == entity and p["start"] - last_end <= 1:
            if not p["word"].startswith("##"):
                current["word"] += " "
            current["word"] += word_piece
            current["score"].append(score)
        else:
            if current["entity_group"]:
                current["score"] = sum(current["score"]) / len(current["score"])
                merged.append(current)
            current = {"entity_group": entity, "word": word_piece, "score": [score]}
        last_end = p["end"]

    if current["entity_group"]:
        current["score"] = sum(current["score"]) / len(current["score"])
        merged.append(current)

    return merged


# Test
text = "Jon Snow isnt a bastard, he is a king, the true heir to the throne. he said, winter is coming"
preds = ner_pipe(text)
merged_preds = merge_entities_split(preds)

for p in merged_preds:
    print(f"{p['word'].strip()} → {p['entity_group']} ({p['score']:.3f})")


jon snow → PER (0.995)
winter → PER (0.645)


In [ ]:
def merge_entities_split(preds):
    merged = []
    current = {"entity_group": None, "word": "", "score": []}
    last_end = -1

    for p in preds:
        entity_tag = p.get("entity", p.get("entity_group"))
        word_piece = p["word"].replace("##", "")
        score = float(p["score"])

        if entity_tag == "O" or entity_tag is None:
            if current["entity_group"]:
                current["score"] = sum(current["score"]) / len(current["score"])
                merged.append(current)
                current = {"entity_group": None, "word": "", "score": []}
            continue

        entity = entity_tag.split("-")[-1]
        if current["entity_group"] == entity and p["start"] - last_end <= 1:
            if not p["word"].startswith("##"):
                current["word"] += " "
            current["word"] += word_piece
            current["score"].append(score)
        else:
            if current["entity_group"]:
                current["score"] = sum(current["score"]) / len(current["score"])
                merged.append(current)
            current = {"entity_group": entity, "word": word_piece, "score": [score]}
        last_end = p["end"]

    if current["entity_group"]:
        current["score"] = sum(current["score"]) / len(current["score"])
        merged.append(current)

    return merged


# ---------- Test with thresholds ----------
text = "roy and aaron were just personalities"
preds = ner_pipe(text)
merged_preds = merge_entities_split(preds)

# Set per-entity confidence thresholds
thresholds = {
    "PER": 0.85,
    "LOC": 0.80,
    "ORG": 0.80,
    "MISC": 0.75
}

print("Device set to use cuda:0")
for p in merged_preds:
    if p['score'] >= thresholds.get(p['entity_group'], 0.85):
        print(f"{p['word'].strip()} → {p['entity_group']} ({p['score']:.3f})")


Device set to use cuda:0
roy → PER (0.993)
aaron → PER (0.994)


In [ ]:

from transformers import pipeline

# Load your trained model
model_dir = "./distilbert_ner_model"
ner_pipe = pipeline("token-classification", model=model_dir, aggregation_strategy="simple")

# --- Helper: Merge tokens properly ---
def merge_entities_split(preds):
    merged = []
    current = {"entity_group": None, "word": "", "score": []}
    last_end = -1

    for p in preds:
        entity_tag = p.get("entity", p.get("entity_group"))
        word_piece = p["word"].replace("##", "")
        score = float(p["score"])

        if entity_tag == "O" or entity_tag is None:
            if current["entity_group"]:
                current["score"] = sum(current["score"]) / len(current["score"])
                merged.append(current)
                current = {"entity_group": None, "word": "", "score": []}
            continue

        entity = entity_tag.split("-")[-1]
        if current["entity_group"] == entity and p["start"] - last_end <= 1:
            if not p["word"].startswith("##"):
                current["word"] += " "
            current["word"] += word_piece
            current["score"].append(score)
        else:
            if current["entity_group"]:
                current["score"] = sum(current["score"]) / len(current["score"])
                merged.append(current)
            current = {"entity_group": entity, "word": word_piece, "score": [score]}
        last_end = p["end"]

    if current["entity_group"]:
        current["score"] = sum(current["score"]) / len(current["score"])
        merged.append(current)
    return merged

# --- Tricky Test Sentences ---
test_sentences = [
    "Elon Musk met Sundar Pichai at Google HQ.",
    "Dr. A.P.J. Abdul Kalam inspired millions.",
    "barack obama and joe biden were seen jogging together.",
    "Messi scored, and Ronaldo smiled from the bench.",
    "Taylor Swift and Selena Gomez went on tour.",
    "The CEO, Satya Nadella, joined the meeting late.",
    "Mr. Sherlock Holmes lives on Baker Street.",
    "oprah winfrey interviewed tom cruise yesterday.",
    "Prime Minister Narendra Modi met Rishi Sunak in Delhi.",
    "elon and mark discussed AI safety with sam altman.",
    "Sachin Tendulkar is known as the God of Cricket.",
    "Lady Gaga collaborated with Bradley Cooper again.",
    "The artist formerly known as Prince performed live.",
    "Captain America defeated Thanos easily.",
    "Barbie met Ken at the beach.",
    "The teacher, Mrs. Fernandez, praised John Doe for his project.",
    "Queen Elizabeth II ruled for decades.",
    "Ariana met The Weeknd during the concert.",
    "King Charles addressed the nation today.",
    "Serena Williams defeated Naomi Osaka in straight sets.",
    "George and Amal Clooney hosted a charity event.",
    "Jeff Bezos visited Elon in Texas.",
    "Zayn Malik performed with Gigi Hadid cheering from the crowd.",
    "Ratan Tata met Elon Musk to discuss EVs.",
    "Mr. Bean isn’t a real person, but Rowan Atkinson is."
]

# --- Run NER Inference ---
for idx, sentence in enumerate(test_sentences, 1):
    preds = ner_pipe(sentence)
    merged = merge_entities_split(preds)
    print(f"\n🧠 Sentence {idx}: {sentence}")
    if not merged:
        print("→ No entities detected.")
    else:
        for p in merged:
            print(f"→ {p['word'].strip()} → {p['entity_group']} ({p['score']:.3f})")


Device set to use cuda:0



🧠 Sentence 1: Elon Musk met Sundar Pichai at Google HQ.
→ elon musk → PER (0.999)
→ sundar pichai → PER (0.999)

🧠 Sentence 2: Dr. A.P.J. Abdul Kalam inspired millions.
→ a p j . abdul kalam → PER (0.957)

🧠 Sentence 3: barack obama and joe biden were seen jogging together.
→ barack obama → PER (0.990)
→ joe biden → PER (0.997)

🧠 Sentence 4: Messi scored, and Ronaldo smiled from the bench.
→ messi → PER (0.976)
→ ronaldo → PER (0.705)

🧠 Sentence 5: Taylor Swift and Selena Gomez went on tour.
→ taylor swift → PER (0.997)
→ selena gomez → PER (0.997)

🧠 Sentence 6: The CEO, Satya Nadella, joined the meeting late.
→ satya nadella → PER (0.999)

🧠 Sentence 7: Mr. Sherlock Holmes lives on Baker Street.
→ sherlock holmes → PER (0.848)

🧠 Sentence 8: oprah winfrey interviewed tom cruise yesterday.
→ oprah winfrey → PER (0.997)
→ tom cruise → PER (0.999)

🧠 Sentence 9: Prime Minister Narendra Modi met Rishi Sunak in Delhi.
→ narendra modi → PER (0.999)
→ rishi sunak → PER (0.999)

🧠 Sentenc

In [ ]:
!zip -r /content/ner_model.zip /content/distilbert_ner_model/


  adding: content/distilbert_ner_model/ (stored 0%)
  adding: content/distilbert_ner_model/vocab.txt (deflated 53%)
  adding: content/distilbert_ner_model/special_tokens_map.json (deflated 42%)
  adding: content/distilbert_ner_model/tokenizer_config.json (deflated 75%)
  adding: content/distilbert_ner_model/tokenizer.json (deflated 71%)
  adding: content/distilbert_ner_model/config.json (deflated 46%)
  adding: content/distilbert_ner_model/model.safetensors (deflated 8%)


In [ ]:
from google.colab import files
files.download("/content/ner_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>